# Sales Forecasting Analysis

This notebook contains my individual contribution to a time series forecasting project using retail sales data.

My contribution starts from Exploratory Data Analysis and continues through model selection, forecasting, and evaluation.

Focus areas:
- exploratory data analysis (EDA)
- identifying key demand patterns
- selecting and evaluating forecasting models
- generating and comparing forecasts

Note: this notebook is a cleaned portfolio version derived from my own contribution in a larger university group project.


## Exploratory Data Analysis
### Section 1


In [ ]:
## Aggregate Sales Dynamics ------------------------------------
total_daily_sales <- full_train %>%
  group_by(date) %>%
  summarise(
    total_sales = sum(sales, na.rm = TRUE)
  ) %>%
  arrange(date)

# Quick check
head(total_daily_sales)
summary(total_daily_sales$total_sales)

ggplot(total_daily_sales, aes(x = date, y = total_sales)) +
  geom_line() +
  labs(
    title = "Total Daily Sales Over Time (TX3 – Food3)",
    x = "Date",
    y = "Units Sold"
  ) +
  theme_minimal()


In [ ]:
ggplot(total_daily_sales, aes(x = date, y = total_sales)) +
  geom_line(alpha = 0.4) +
  geom_smooth(method = "loess", span = 0.2) +
  labs(
    title = "Total Daily Sales with Smoothed Trend",
    x = "Date",
    y = "Units Sold"
  ) +
  theme_minimal()


In [ ]:

## Distribution of Daily Item-Level Sales --------------------
ggplot(full_train, aes(x = sales)) +
  geom_histogram(bins = 50, fill = "steelblue", alpha = 0.7) +
  labs(
    title = "Distribution of Daily Item-Level Sales",
    x = "Units Sold",
    y = "Frequency"
  ) +
  theme_minimal()

full_train %>%
  summarise(
    pct_zero_sales = mean(sales == 0),
    mean_sales     = mean(sales),
    median_sales   = median(sales)
  )


In [ ]:
zero_by_item <- full_train %>%
  group_by(item_id) %>%
  summarise(pct_zero_sales = mean(sales == 0))

ggplot(zero_by_item, aes(x = pct_zero_sales)) +
  geom_histogram(bins = 30, fill = "darkorange", alpha = 0.7) +
  labs(
    title = "Distribution of Zero-Sales Days Across Items",
    x = "Proportion of Zero-Sales Days",
    y = "Number of Items"
  ) +
  theme_minimal()


In [ ]:
mean_var <- full_train %>%
  group_by(item_id) %>%
  summarise(
    mean_sales = mean(sales),
    var_sales  = var(sales)
  )

ggplot(mean_var, aes(x = mean_sales, y = var_sales)) +
  geom_point(alpha = 0.4) +
  scale_x_log10() +
  scale_y_log10() +
  labs(
    title = "Mean–Variance Relationship at Item Level",
    x = "Mean Sales (log scale)",
    y = "Variance of Sales (log scale)"
  ) +
  theme_minimal()


In [ ]:
## Weekday Effects -------------------------------------------
weekday_sales <- full_train %>%
  group_by(weekday) %>%
  summarise(avg_sales = mean(sales, na.rm = TRUE))

ggplot(weekday_sales, aes(x = weekday, y = avg_sales)) +
  geom_col(fill = "steelblue", alpha = 0.8) +
  labs(
    title = "Average Daily Sales by Weekday",
    x = "Weekday",
    y = "Average Units Sold"
  ) +
  theme_minimal()


### Section 2


In [ ]:
## Monthly Seasonality ---------------------------------------

monthly_sales <- full_train %>%
  group_by(month) %>%
  summarise(avg_sales = mean(sales, na.rm = TRUE))

ggplot(monthly_sales, aes(x = factor(month), y = avg_sales)) +
  geom_col(fill = "darkgreen", alpha = 0.8) +
  labs(
    title = "Average Daily Sales by Month",
    x = "Month",
    y = "Average Units Sold"
  ) +
  theme_minimal()


In [ ]:
## Event Effects ---------------------------------------------
event_sales <- full_train %>%
  mutate(event = ifelse(is.na(event_name_1), "No Event", "Event")) %>%
  group_by(event) %>%
  summarise(avg_sales = mean(sales, na.rm = TRUE))

ggplot(event_sales, aes(x = event, y = avg_sales)) +
  geom_col(fill = "purple", alpha = 0.8) +
  labs(
    title = "Average Sales on Event vs Non-Event Days",
    x = "",
    y = "Average Units Sold"
  ) +
  theme_minimal()


In [ ]:
## SNAP Effects ----------------------------------------------
snap_sales <- full_train %>%
  group_by(snap_TX) %>%
  summarise(avg_sales = mean(sales, na.rm = TRUE))

ggplot(snap_sales, aes(x = factor(snap_TX), y = avg_sales)) +
  geom_col(fill = "darkorange", alpha = 0.8) +
  labs(
    title = "Effect of SNAP Eligibility on Sales",
    x = "SNAP_TX (0 = No, 1 = Yes)",
    y = "Average Units Sold"
  ) +
  theme_minimal()


## Feature Engineering


In [ ]:
### Calendar & Price Features
fe_data <- full_train %>%
  mutate(
    # Calendar features
    weekday = factor(
      weekday,
      levels = c("Saturday", "Sunday", "Monday", "Tuesday",
                 "Wednesday", "Thursday", "Friday")
    ),
    month = factor(month),

    # Event indicator
    event_flag = ifelse(is.na(event_name_1), 0, 1),

    # SNAP already binary, just ensure type
    snap_TX = factor(snap_TX)
  )

glimpse(fe_data)


In [ ]:
fe_data <- fe_data %>%
  group_by(item_id, store_id) %>%
  arrange(date) %>%
  mutate(
    # Price change (week-to-week)
    price_change = sell_price - lag(sell_price),

    # Relative price (normalized by item mean)
    price_relative = ifelse(
      is.na(sell_price),
      NA,
      sell_price / mean(sell_price, na.rm = TRUE)
    )

  ) %>%
  ungroup()

summary(fe_data$price_change)
summary(fe_data$price_relative)


In [ ]:
# Check missingness
fe_data %>%
  summarise(
    pct_missing_price = mean(is.na(sell_price)),
    pct_missing_price_change = mean(is.na(price_change))
  )

# Check factors
levels(fe_data$weekday)
levels(fe_data$month)
levels(fe_data$snap_TX)


In [ ]:
### Lag & Rolling Features
fe_data <- fe_data %>%
  group_by(item_id, store_id) %>%
  arrange(date) %>%
  mutate(
    lag_7  = lag(sales, 7),
    lag_14 = lag(sales, 14),
    lag_28 = lag(sales, 28)
  ) %>%
  ungroup()

summary(fe_data$lag_7)
summary(fe_data$lag_28)


In [ ]:
fe_data <- fe_data %>%
  group_by(item_id, store_id) %>%
  arrange(date) %>%
  mutate(
    rolling_mean_7 = slide_dbl(
      sales,
      mean,
      .before = 7,
      .after  = -1,
      na.rm   = TRUE
    ),
    rolling_mean_28 = slide_dbl(
      sales,
      mean,
      .before = 28,
      .after  = -1,
      na.rm   = TRUE
    )
  ) %>%
  ungroup()

summary(fe_data$rolling_mean_7)
summary(fe_data$rolling_mean_28)


In [ ]:
# How many rows have all lag features available?
fe_data %>%
  summarise(
    pct_complete_lags = mean(
      !is.na(lag_7) & !is.na(lag_14) & !is.na(lag_28)
    )
  )

# Check alignment visually for one item
fe_data %>%
  filter(item_id == first(item_id)) %>%
  select(date, sales, lag_7, rolling_mean_7) %>%
  head(15)


## Model Selection
### Preperation


In [ ]:
to_long_sales <- function(df_wide) {
  df_wide %>%
    pivot_longer(cols = starts_with("d_"), names_to = "d", values_to = "sales") %>%
    separate(id, into = c("cat", "dept", "item_num", "state", "store_num", "tag"),
             sep = "_", remove = FALSE) %>%
    mutate(
      item_id  = paste(cat, dept, item_num, sep = "_"),
      store_id = paste(state, store_num, sep = "_")
    )
}


In [ ]:
sales_val_wide  <- read.csv("Final Project Datasets/sales_test_validation_afcs2025.csv")
sales_eval_wide <- read.csv("Final Project Datasets/sales_test_evaluation_afcs_2025.csv")

val_long  <- to_long_sales(sales_val_wide)  %>% left_join(calendar, by = "d") %>%
  left_join(sell_prices, by = c("item_id","store_id","wm_yr_wk"))

eval_long <- to_long_sales(sales_eval_wide) %>% left_join(calendar, by = "d") %>%
  left_join(sell_prices, by = c("item_id","store_id","wm_yr_wk"))


In [ ]:
history_train <- full_train %>%
  mutate(date = as.Date(date)) %>%
  select(id, item_id, store_id, d, date, sales, weekday, month, year,
         event_name_1, snap_TX, wm_yr_wk, sell_price)
### feauture engineering
future_val  <- val_long  %>% mutate(date = as.Date(date))
future_eval <- eval_long %>% mutate(date = as.Date(date))


### Classical Model: Seasonal Naive


In [ ]:
ts_train <- history_train %>%
  group_by(id) %>%
  arrange(date) %>%
  as_tsibble(key = id, index = date)

fit_snaive <- ts_train %>%
  model(snaive = SNAIVE(sales ~ lag("week")))

fc_val_snaive <- forecast(fit_snaive, h = 28) %>%
  as_tibble() %>%
  select(id, date, .mean) %>%
  rename(pred = .mean)

val_rmse_snaive <- future_val %>%
  select(id, date, sales) %>%
  inner_join(fc_val_snaive, by = c("id","date")) %>%
  summarise(rmse = sqrt(mean((sales - pred)^2, na.rm = TRUE)))

val_rmse_snaive


In [ ]:
fit_ets <- ts_train %>%
  model(ets = ETS(sales))

fc_val_ets <- forecast(fit_ets, h = 28) %>%
  as_tibble() %>%
  select(id, date, .mean) %>%
  rename(pred = .mean)

val_rmse_ets <- future_val %>%
  select(id, date, sales) %>%
  inner_join(fc_val_ets, by = c("id","date")) %>%
  summarise(rmse = sqrt(mean((sales - pred)^2, na.rm = TRUE)))

val_rmse_ets


### Classical Model: ETS


In [ ]:
ts_train <- history_train %>%
  group_by(id) %>%
  arrange(date) %>%
  as_tsibble(key = id, index = date)

fit_snaive <- ts_train %>%
  model(snaive = SNAIVE(sales ~ lag("week")))

fc_val_snaive <- forecast(fit_snaive, h = 28) %>%
  as_tibble() %>%
  select(id, date, .mean) %>%
  rename(pred = .mean)

val_rmse_snaive <- future_val %>%
  select(id, date, sales) %>%
  inner_join(fc_val_snaive, by = c("id","date")) %>%
  summarise(rmse = sqrt(mean((sales - pred)^2, na.rm = TRUE)))

val_rmse_snaive


## Feature Engineering for ML (XGBoost + LightGBM)


In [ ]:
make_ml_features <- function(df) {
  df %>%
    group_by(id) %>%
    arrange(date) %>%
    mutate(
      weekday = factor(weekday, levels = c("Saturday","Sunday","Monday","Tuesday","Wednesday","Thursday","Friday")),
      month   = factor(month),
      snap_TX = factor(snap_TX),
      event_flag = ifelse(is.na(event_name_1), 0, 1),

      lag_7  = lag(sales, 7),
      lag_14 = lag(sales, 14),
      lag_28 = lag(sales, 28),

      roll_mean_7  = slide_dbl(sales, mean, .before = 7,  .after = -1, na.rm = TRUE),
      roll_mean_28 = slide_dbl(sales, mean, .before = 28, .after = -1, na.rm = TRUE)
    ) %>%
    ungroup()
}

train_fe <- make_ml_features(history_train) %>%
  filter(!is.na(lag_28)) %>%
  mutate(
    wday = as.integer(weekday),
    mon  = as.integer(month),
    snap = as.integer(as.character(snap_TX))
  )

X_train <- train_fe %>%
  transmute(
    lag_7, lag_14, lag_28,
    roll_mean_7, roll_mean_28,
    sell_price, event_flag, snap, wday, mon
  ) %>% as.matrix()

y_train <- train_fe$sales


## ML Model: XGBoost (Validation)


In [ ]:
dtrain <- xgb.DMatrix(data = X_train, label = y_train)

params_xgb <- list(
  objective = "reg:squarederror",
  eta = 0.05,
  max_depth = 8,
  subsample = 0.8,
  colsample_bytree = 0.8
)

xgb_model <- xgb.train(
  params = params_xgb,
  data = dtrain,
  nrounds = 500,
  verbose = 0
)

recursive_forecast_xgb <- function(model, history_df, future_df, horizon = 28) {
  out <- list()
  current <- history_df

  future_dates <- sort(unique(future_df$date))

  for (k in 1:horizon) {
    target_date <- future_dates[k]

    next_day <- future_df %>%
      filter(date == target_date) %>%
      select(names(current)) %>%
      mutate(sales = NA_real_)

    current_ext <- bind_rows(current, next_day)
    current_fe  <- make_ml_features(current_ext)

    pred_rows <- current_fe %>%
      filter(date == target_date) %>%
      mutate(
        wday = as.integer(weekday),
        mon  = as.integer(month),
        snap = as.integer(as.character(snap_TX))
      )

    X_pred <- pred_rows %>%
      transmute(
        lag_7, lag_14, lag_28,
        roll_mean_7, roll_mean_28,
        sell_price, event_flag, snap, wday, mon
      ) %>% as.matrix()

    dpred <- xgb.DMatrix(X_pred)
    preds <- predict(model, dpred)

    pred_rows$pred <- pmax(preds, 0)
    out[[k]] <- pred_rows %>% select(id, date, pred)

    current <- current_ext %>%
      left_join(out[[k]], by = c("id","date")) %>%
      mutate(sales = ifelse(is.na(sales) & !is.na(pred), pred, sales)) %>%
      select(-pred)
  }

  bind_rows(out)
}

pred_val_xgb <- recursive_forecast_xgb(xgb_model, history_train, future_val, horizon = 28)

val_rmse_xgb <- future_val %>%
  select(id, date, sales) %>%
  inner_join(pred_val_xgb, by = c("id","date")) %>%
  summarise(rmse = sqrt(mean((sales - pred)^2, na.rm = TRUE)))

val_rmse_xgb


## ML Model: LightGBM (Validation)


In [ ]:
lgb_train <- lgb.Dataset(data = X_train, label = y_train)

params_lgb <- list(
  objective = "regression",
  metric = "rmse",
  learning_rate = 0.05,
  num_leaves = 31,
  max_depth = -1,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq = 5,
  verbosity = -1
)

lgb_model <- lgb.train(
  params = params_lgb,
  data = lgb_train,
  nrounds = 500,
  verbose = -1
)

recursive_forecast_lgb <- function(model, history_df, future_df, horizon = 28) {
  out <- list()
  current <- history_df
  future_dates <- sort(unique(future_df$date))

  for (k in 1:horizon) {
    target_date <- future_dates[k]

    next_day <- future_df %>%
      filter(date == target_date) %>%
      select(names(current)) %>%
      mutate(sales = NA_real_)

    current_ext <- bind_rows(current, next_day)
    current_fe  <- make_ml_features(current_ext)

    pred_rows <- current_fe %>%
      filter(date == target_date) %>%
      mutate(
        wday = as.integer(weekday),
        mon  = as.integer(month),
        snap = as.integer(as.character(snap_TX))
      )

    X_pred <- pred_rows %>%
      transmute(
        lag_7, lag_14, lag_28,
        roll_mean_7, roll_mean_28,
        sell_price, event_flag, snap, wday, mon
      ) %>% as.matrix()

    preds <- predict(model, X_pred)
    pred_rows$pred <- pmax(preds, 0)

    out[[k]] <- pred_rows %>% select(id, date, pred)

    current <- current_ext %>%
      left_join(out[[k]], by = c("id","date")) %>%
      mutate(sales = ifelse(is.na(sales) & !is.na(pred), pred, sales)) %>%
      select(-pred)
  }

  bind_rows(out)
}

pred_val_lgb <- recursive_forecast_lgb(lgb_model, history_train, future_val, horizon = 28)

val_rmse_lgb <- future_val %>%
  select(id, date, sales) %>%
  inner_join(pred_val_lgb, by = c("id","date")) %>%
  summarise(rmse = sqrt(mean((sales - pred)^2, na.rm = TRUE)))

val_rmse_lgb


In [ ]:
# 1) Aggregate actuals (already 28 rows)
agg_val <- future_val %>%
  group_by(date) %>%
  summarise(actual = sum(sales, na.rm = TRUE)) %>%
  arrange(date)

# 2) Aggregate predictions to daily totals (also 28 rows)
pred_val_lgb_daily <- pred_val_lgb %>%
  group_by(date) %>%
  summarise(predicted = sum(pred, na.rm = TRUE)) %>%
  arrange(date)

# 3) Join and plot
lgb_df <- agg_val %>%
  left_join(pred_val_lgb_daily, by = "date")

ggplot(lgb_df, aes(x = date)) +
  geom_line(aes(y = actual, color = "Actual")) +
  geom_line(aes(y = predicted, color = "LightGBM Forecast")) +
  labs(
    title = "Aggregated Sales: LightGBM vs Actual (Validation)",
    y = "Units Sold",
    color = ""
  ) +
  theme_minimal()


## DL Model


In [ ]:
# --- 1) Aggregate TRAIN to daily total ---
agg_train <- history_train %>%
  group_by(date) %>%
  summarise(y = sum(sales, na.rm = TRUE)) %>%
  arrange(date)

# --- 2) Aggregate VALIDATION actuals (already in your code, but keeping consistent) ---
agg_val <- future_val %>%
  group_by(date) %>%
  summarise(actual = sum(sales, na.rm = TRUE)) %>%
  arrange(date)

# --- 3) Scale data using TRAIN statistics ONLY (no leakage) ---
y_train <- agg_train$y
mu <- mean(y_train); sig <- sd(y_train)
y_train_s <- (y_train - mu) / sig

# --- 4) Create sliding windows for supervised learning ---
window <- 28
N <- length(y_train_s) - window

X <- array(NA_real_, dim = c(N, window, 1))
Y <- array(NA_real_, dim = c(N, 1))

for (i in 1:N) {
  X[i, , 1] <- y_train_s[i:(i + window - 1)]
  Y[i, 1]   <- y_train_s[i + window]
}

X_t <- torch_tensor(X)
Y_t <- torch_tensor(Y)

# --- 5) Define LSTM model (lightweight) ---
lstm_net <- nn_module(
  initialize = function(input_size = 1, hidden_size = 32, num_layers = 1, dropout = 0.0) {
    self$lstm <- nn_lstm(
      input_size = input_size,
      hidden_size = hidden_size,
      num_layers = num_layers,
      batch_first = TRUE,
      dropout = dropout
    )
    self$fc <- nn_linear(hidden_size, 1)
  },
  forward = function(x) {
    out <- self$lstm(x)[[1]]           # (batch, seq, hidden)
    last <- out[ , dim(out)[2], ]      # last time step (batch, hidden)
    self$fc(last)                      # (batch, 1)
  }
)

model <- lstm_net()
opt <- optim_adam(model$parameters, lr = 1e-3)
loss_fn <- nn_mse_loss()

# --- 6) Train on TRAIN only ---
set.seed(123)

epochs <- 15
batch_size <- 64

for (ep in 1:epochs) {
  model$train()
  idx <- sample.int(N)
  total_loss <- 0

  for (start in seq(1, N, by = batch_size)) {
    end <- min(start + batch_size - 1, N)
    b <- idx[start:end]

    xb <- X_t[b,..]
    yb <- Y_t[b,..]

    opt$zero_grad()
    pred <- model(xb)
    loss <- loss_fn(pred, yb)
    loss$backward()
    opt$step()

    total_loss <- total_loss + loss$item()
  }

  if (ep %% 5 == 0) cat("epoch", ep, "loss", total_loss, "\n")
}

# --- 7) Recursive 28-day forecast for VALIDATION horizon ---
model$eval()

last_window <- tail(y_train_s, window)
pred_s <- numeric(28)

for (h in 1:28) {
  x_in <- array(last_window, dim = c(1, window, 1))
  x_in_t <- torch_tensor(x_in)
  yhat <- as.numeric(model(x_in_t)$to(device = "cpu"))
  pred_s[h] <- yhat
  last_window <- c(last_window[-1], yhat)
}

pred_lstm_val <- pred_s * sig + mu
rmse_lstm <- sqrt(mean((agg_val$actual - pred_lstm_val)^2, na.rm = TRUE))
rmse_lstm


In [ ]:
lstm_df <- tibble(date = agg_val$date, actual = agg_val$actual, predicted = pred_lstm_val)

ggplot(lstm_df, aes(x = date)) +
  geom_line(aes(y = actual, color = "Actual")) +
  geom_line(aes(y = predicted, color = "LSTM Forecast")) +
  labs(title = "Aggregated Sales: LSTM vs Actual (Validation)", y = "Units Sold", color = "") +
  theme_minimal()


In [ ]:
rmse_table <- tibble(
  model = c("Seasonal Naive", "ETS", "XGBoost", "LightGBM", "LSTM"),
  rmse_validation = c(
    val_rmse_snaive$rmse,
    val_rmse_ets$rmse,
    val_rmse_xgb$rmse,
    val_rmse_lgb$rmse,
    rmse_lstm  
  )
) %>% arrange(rmse_validation)

rmse_table


In [ ]:
pred_eval_xgb <- recursive_forecast_xgb(
  model      = xgb_model,
  history_df = history_train,
  future_df  = future_eval,
  horizon    = 28
)

# standardize name
pred_eval <- pred_eval_xgb %>% rename(pred = pred)

# --- Final RMSE on evaluation set ---
final_rmse <- future_eval %>%
  select(id, date, sales) %>%
  inner_join(pred_eval, by = c("id","date")) %>%
  summarise(rmse = sqrt(mean((sales - pred)^2, na.rm = TRUE)))

final_rmse


### 3) One evaluation plot (for the report)


In [ ]:
# --- One evaluation plot (aggregate) ---
agg_eval <- future_eval %>%
  group_by(date) %>%
  summarise(actual = sum(sales, na.rm = TRUE)) %>%
  left_join(
    pred_eval %>%
      group_by(date) %>%
      summarise(predicted = sum(pred, na.rm = TRUE)),
    by = "date"
  )

ggplot(agg_eval, aes(x = date)) +
  geom_line(aes(y = actual, color = "Actual")) +
  geom_line(aes(y = predicted, color = "Forecast")) +
  labs(
    title = "Aggregate Sales: Actual vs XGBoost Forecast (Evaluation)",
    y = "Units Sold",
    color = ""
  ) +
  theme_minimal()


In [ ]:
sample_sub <- read.csv("Final Project Datasets/sample_submission_afcs2025.csv")

submission_wide <- pred_eval %>%
  mutate(h = as.integer(date - min(date)) + 1L) %>%
  select(id, h, pred) %>%
  mutate(h = paste0("F", h)) %>%
  pivot_wider(
    names_from  = h,
    values_from = pred
  )

# Match sample submission ordering/ids
submission <- sample_sub %>%
  select(id) %>%
  left_join(submission_wide, by = "id")

# Write final file
write.csv(
  submission,
  "Team01_FinalSubmission_bestmodel.csv",
  row.names = FALSE
)

head(submission)
